# Task 2: Predicting severity metrics
Run this notebook to predict the severity (UPDRSM). 

The structure of this notebook is as follows: 
1. Data setup: loading features, metadata, splits
2. Training helper: function to train models and save outputs
3. Mean baseline 
4. Multiple linear regression
5. Random forest
6. 1D-CNN

## 1. Data setup

In [1]:
# imports and setting file paths
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.dummy import DummyRegressor
import tensorflow as tf

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SPLITS_PATH = ROOT / "data" / "splits.json"

# output directory
OUT = ROOT / "presentation_material" / "task2"
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
(OUT / "weights").mkdir(parents=True, exist_ok=True)

# import functions from models 
from src.data import load_metadata, col_names
from src.features import X as features
from src.models.linreg import make_linear_regression
from src.models.rf import make_random_forest
from src.models.cnn1d import build_cnn

print("Features shape:", features.shape)
display(features.head())

        stride_time  stride_cv  swing_cv  swing_asymmetry  peak_force
GaCo01     1.230322   0.042125  0.054223         0.013902   14.561747
GaCo02     1.103386   0.035667  0.037527         0.001486   15.233429
GaCo03     1.241191   0.049523  0.078598         0.103336   12.754554
GaCo04     1.249965   0.058603  0.064637         0.022956   15.401692
GaCo05     1.051882   0.039561  0.055866         0.013298   12.820625
...             ...        ...       ...              ...         ...
SiPt36     1.065997   0.041058  0.077181         0.066450   14.236129
SiPt37     1.210989   0.056895  0.072182         0.011100   12.766210
SiPt38     1.163753   0.039291  0.049182         0.000858   12.365833
SiPt39     1.067753   0.048137  0.056888         0.065241   21.129340
SiPt40     1.005855   0.045557  0.056307         0.018599   13.568025

[165 rows x 5 columns]
GaCo01    0
GaCo02    0
GaCo03    0
GaCo04    0
GaCo05    0
         ..
SiPt36    1
SiPt37    1
SiPt38    1
SiPt39    1
SiPt40    1
Name

,stride_time,stride_cv,swing_cv,swing_asymmetry,peak_force
GaCo01,1.230322,0.042125,0.054223,0.013902,14.561747
GaCo02,1.103386,0.035667,0.037527,0.001486,15.233429
GaCo03,1.241191,0.049523,0.078598,0.103336,12.754554
GaCo04,1.249965,0.058603,0.064637,0.022956,15.401692
GaCo05,1.051882,0.039561,0.055866,0.013298,12.820625


In [2]:
# load data 
metadata = load_metadata().rename(columns={'ID':'subject'})
features = features.rename_axis('subject').reset_index()

# merge metadata and features into patients 
patients = features.merge(
    metadata[['subject','Group','UPDRSM']], 
    on='subject', how='inner', validate='many_to_one')
patients = patients.loc[
    patients['Group'].eq('Pt') & patients['UPDRSM'].notna()].reset_index(drop=True)

feature_columns = ['stride_time','stride_cv','swing_cv','swing_asymmetry','peak_force']

X = patients[feature_columns]
y = patients['UPDRSM'].astype(float)

print("Patients:", patients["subject"].nunique(), "| Feature matrix:", X.shape)
display(X.head())

Patients: 91 | Feature matrix: (91, 5)


,stride_time,stride_cv,swing_cv,swing_asymmetry,peak_force
0,1.520570,0.076831,0.128824,0.011780,15.297700
1,1.286201,0.054108,0.093835,0.082978,13.863056
2,1.079744,0.054986,0.077635,0.079694,17.089902
3,1.158036,0.046661,0.060210,0.083254,14.352317
4,1.380191,0.088105,0.139013,0.254944,14.187963


In [3]:
with open(SPLITS_PATH) as f:
    assignment = json.load(f)["task2"]

test_subjects = set(assignment["test"])
folds = []
for fold in assignment["folds"]:
    train_ids, val_ids = set(fold["train"]), set(fold["val"])
    
    # safety check: keep the held-out test subjects separate.
    assert train_ids.isdisjoint(val_ids | test_subjects) and val_ids.isdisjoint(test_subjects)
    train_idx = np.flatnonzero(patients["subject"].isin(train_ids).to_numpy())
    val_idx = np.flatnonzero(patients["subject"].isin(val_ids).to_numpy())
    folds.append((train_idx, val_idx))

print('CV folds:',len(folds),'held-out test subjects:',len(test_subjects))
print("Using", len(folds), "saved validation folds; held-out test subjects untouched.")

CV folds: 5 held-out test subjects: 19
Using 5 saved validation folds; held-out test subjects untouched.


## 2. Training helper

Function to implement at every model to train the model and save the output. 

In [4]:
def train_and_save(model_name, make_model):
    for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):
        model = make_model()  
        model.fit(X.iloc[train_idx], y.iloc[train_idx])  # train only on training subjects
        predictions = np.asarray(model.predict(X.iloc[val_idx])).reshape(-1)

        # Save actual values and predictions for the results teammate.
        output = pd.DataFrame({
            "subject": patients.iloc[val_idx]["subject"].to_numpy(),
            "fold": fold_number,
            "actual_UPDRSM": y.iloc[val_idx].to_numpy(),
            "predicted_UPDRSM": predictions
        })
        output.to_csv(
            OUT / "predictions" / f"{model_name}_val_fold{fold_number}.csv",
            index=False
        )
        # save fitted preprocessing steps and model together
        joblib.dump(
            model, OUT / "weights" / f"{model_name}_val_fold{fold_number}.joblib"
        )
        print(f"{model_name}: saved validation fold {fold_number} ({len(output)} rows)")
    print(f"{model_name}: model train complete")

## 3. Mean baseline 
This is used as a reference point. Ideally, our models should perform better than this. We predict the training-fold mean UPDRSM for each validation subject.

In [5]:
train_and_save("mean_baseline", lambda: DummyRegressor(strategy="mean"))

mean_baseline: saved validation fold 1 (15 rows)
mean_baseline: saved validation fold 2 (15 rows)
mean_baseline: saved validation fold 3 (14 rows)
mean_baseline: saved validation fold 4 (14 rows)
mean_baseline: saved validation fold 5 (14 rows)
mean_baseline: model train complete


## 4. Multiple Linear Regression

In [6]:
train_and_save("linear_regression", make_linear_regression)

linear_regression: saved validation fold 1 (15 rows)
linear_regression: saved validation fold 2 (15 rows)
linear_regression: saved validation fold 3 (14 rows)
linear_regression: saved validation fold 4 (14 rows)
linear_regression: saved validation fold 5 (14 rows)
linear_regression: model train complete


## 5. Random Forest

In [7]:
train_and_save("random_forest", make_random_forest)

random_forest: saved validation fold 1 (15 rows)
random_forest: saved validation fold 2 (15 rows)
random_forest: saved validation fold 3 (14 rows)
random_forest: saved validation fold 4 (14 rows)
random_forest: saved validation fold 5 (14 rows)
random_forest: model train complete


explain reason behind each parameter choice? 

In [ ]:
# hyperparameter tuning 

from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error
from sklearn.base import clone

# combinations to test
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5, None],
    "model__min_samples_leaf": [2, 5]
}

tuning_results = [] # store results

for params in ParameterGrid(param_grid):
    fold_rmse = []
    
    for train_idx, val_idx in folds:
        # create new model and apply parameter combination 
        model = make_random_forest() 
        model.set_params(**params) 
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        predictions = model.predict(X.iloc[val_idx])

        # calculate validation RMSE
        rmse = np.sqrt(mean_squared_error(y.iloc[val_idx],predictions))

        fold_rmse.append(rmse)

    # calculate average validation RMSE across five folds.
    tuning_results.append({
        **params,
        "mean_RMSE": np.mean(fold_rmse),
        "std_RMSE": np.std(fold_rmse)
    })

rf_tuning = pd.DataFrame(tuning_results)
rf_tuning = rf_tuning.sort_values("mean_RMSE")
display(rf_tuning)

print("Best RF hyperparameters:")
display(rf_tuning.head(1))


## 6. 1D-CNN Regression
The CNN model uses 16 force-sensor channels over 1,000 time steps per subject. This is a 10-second starting window. 

In [8]:
PROCESSED = ROOT/'data'/'processed'
RAW = ROOT/'data'/'raw'
TIME_STEPS = 1000
SENSOR_COLS = [f'L{i}' for i in range(1,9)] + [f'R{i}' for i in range(1,9)]

def load_signal(subject):
    csv_path = PROCESSED/f'{subject}_01.csv'
    txt_path = RAW/f'{subject}_01.txt'
    if csv_path.exists():
        walk = pd.read_csv(csv_path)
    else:
        walk = pd.read_csv(txt_path,sep=r'\s+',header=None,names=col_names)
    signal = walk[SENSOR_COLS].iloc[:TIME_STEPS].to_numpy(dtype=np.float32)
    if signal.shape != (TIME_STEPS,16) or not np.isfinite(signal).all():
        raise ValueError(f'Invalid CNN signal for {subject}: {signal.shape}')
    return signal

signals = np.stack([load_signal(subject) for subject in patients['subject']])
print('CNN input shape (subjects, time steps, channels):',signals.shape)

CNN input shape (subjects, time steps, channels): (91, 1000, 16)


In [ ]:
# train CNN model per fold, saving predictions and weights for each
import tensorflow as tf
tf.keras.utils.set_random_seed(42)
EPOCHS = 20
BATCH_SIZE = 16
for fold_number, (train_idx, val_idx) in enumerate(folds,1):
    # normalise signals 
    mean = signals[train_idx].mean(axis=(0,1),keepdims=True)
    std = np.maximum(signals[train_idx].std(axis=(0,1),keepdims=True),1e-6)
    x_train = (signals[train_idx]-mean)/std
    x_val = (signals[val_idx]-mean)/std
    
    # create and train model for each fold 
    model = build_cnn(input_shape=(TIME_STEPS,16),
                      task='regression')
    model.fit(x_train,y.iloc[train_idx].to_numpy(),
              epochs=EPOCHS,batch_size=BATCH_SIZE,verbose=1)
    
    # predict with test data 
    predicted = model.predict(x_val,verbose=0).reshape(-1)
    pd.DataFrame({'subject':patients.iloc[val_idx]['subject'].to_numpy(),
                  'fold':fold_number,'actual_UPDRSM':y.iloc[val_idx].to_numpy(),
                  'predicted_UPDRSM':predicted}).to_csv(
        OUT/'predictions'/f'cnn_val_fold{fold_number}.csv',index=False)
    model.save(OUT/'weights'/f'cnn_val_fold{fold_number}.keras')
    
    # save predictions and model weights
    np.savez(OUT/'weights'/f'cnn_val_fold{fold_number}_normalisation.npz',
             mean=mean,std=std,time_steps=TIME_STEPS,sensor_columns=np.array(SENSOR_COLS))
    print(f'Saved CNN fold {fold_number} ({len(val_idx)} predictions)')
print("CNN-1D model training complete.")

# should add plot of training accuracy and validation accuracy? - see lecture code 

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 410.5540 - mae: 18.5496  
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 383.5503 - mae: 17.7956 
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 352.5683 - mae: 16.8850
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 314.0274 - mae: 15.6758 
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 266.4514 - mae: 14.0876 
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 211.2264 - mae: 12.1071 
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 153.8764 - mae: 9.7495 
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 106.0469 - mae: 7.6712
Epoch 9/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 82.7157 - mae: 7.1103 
Epoch 10/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 84.1473 - mae: 7.5378 
Epoch 11/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 86.7039 - mae: 7.7381 
Epoch 12/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 80.5525 - mae: 7.3755 
Epoch 13/20
4/4 ━━━━━━━━━━━━━━━━━━━

In [ ]:
# hyperparameter tuning 
learning_rates = [0.001, 0.0001]
epoch_options = [10, 20, 30]

cnn_tuning_results = []

for learning_rate in learning_rates:
    for epochs in epoch_options:

        fold_rmse = []
        
        for fold_number, (train_idx, val_idx) in enumerate(folds):
            tf.keras.utils.set_random_seed(42)

            # nomralisation
            mean = signals[train_idx].mean(axis=(0, 1), keepdims=True)
            std = np.maximum(signals[train_idx].std(axis=(0, 1), keepdims=True), 1e-6)
            x_train = (signals[train_idx] - mean) / std
            x_val = (signals[val_idx] - mean) / std

            # build and train new cnn
            model = build_cnn(
                input_shape=(TIME_STEPS, 16),
                task="regression",
                learning_rate=learning_rate)
            
            model.fit(x_train, y.iloc[train_idx].to_numpy(),
                epochs=epochs, batch_size=16, verbose=0)

            # generate validation predictions
            predictions = model.predict(x_val, verbose=0).reshape(-1)

            # calculate validation RMSE
            rmse = np.sqrt(mean_squared_error(y.iloc[val_idx], predictions))
            fold_rmse.append(rmse)

        cnn_tuning_results.append({
            "learning_rate": learning_rate,
            "epochs": epochs,
            "mean_RMSE": np.mean(fold_rmse),
            "std_RMSE": np.std(fold_rmse)
        })

# compare configurations
cnn_tuning = pd.DataFrame(cnn_tuning_results)
cnn_tuning = cnn_tuning.sort_values("mean_RMSE")

display(cnn_tuning)

print("Best CNN configuration:")
display(cnn_tuning.head(1))